In [9]:
# Importing Packages required for Dataset Creation.
import json
import requests
import pandas as pd

In [10]:
# Calling openFDA API call with 999 records
open_FDA_API_Resp = requests.get("https://api.fda.gov/drug/event.json?limit=999").json().get("results", [])

In [11]:
# dataset list declaration
dataset = []

In [14]:
#Using Ollama Rest API call to combine 2 models for dataset creation
for i, t in enumerate(open_FDA_API_Resp):
    try:
        drug = t["patient"]["drug"][0]["medicinalproduct"]
        aes = ", ".join([r["reactionmeddrapt"] for r in t["patient"]["reaction"]])

        # MedGemma used for generating the raw and medically rich narrative text
        med_gemma_prompt = f"Write a clinical narrative for {drug} causing {aes}. After the story, add a section '=== EVALUATION ===' with Seriousness, MedDRA PTs, and Naranjo score breakdown."
        med_gemma_resp = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "medgemma1.5:4b", 
                "prompt": med_gemma_prompt, 
                "stream": False
            },
        ).json()
        med_gemma_narrative = med_gemma_resp["response"]

        # Qwen parses the narrative into a strict, validated JSON block
        qwen_json_prompt = f"Analyze: '{med_gemma_narrative}'. Split it into JSON. Key 'clean_narrative' must contain ONLY the raw patient story (completely remove the evaluation section). Keys: 'seriousness' (bool), 'meddra_terms' (list), 'naranjo_score' (int), 'rationale' (text)."
        qwen_json_resp = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "qwen3.5:9b",
                "prompt": qwen_json_prompt,
                "stream": False,
                "format": "json",
            },
        ).json()
        qwen_json_output = json.loads(qwen_json_resp["response"])

        dataset.append({
            "Case_ID": f"PV-HOSP-{i+1:02d}",
            "Input_Narrative": qwen_json_output["clean_narrative"],
            "Structured_Output_JSON": json.dumps({k: v for k, v in qwen_json_output.items() if k != "clean_narrative"}, indent=2)
        })
    except Exception as e:
        print(f"Error details: {e}")

Error details: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
Error details: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
Error details: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
Error details: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection

In [ ]:
# Using Pandas to convert to excel file
pd.DataFrame(dataset).to_excel("med_rev_asst_dataset.xlsx", index=False)
print("Dataset ready for fine-tuning!")

In [8]:
dataset

[]